# Section 1: Email Security and Phishing Detection

This is a **self-contained notebook**. It downloads and parses the dataset, cleans and splits the emails, defines both models, trains them, evaluates them, and saves the results without importing project code from `src/` or calling anything in `scripts/`.

Models compared:

1. TF-IDF with Logistic Regression
2. Token-embedding LSTM

> **Validity warning:** SpamAssassin is labelled spam versus ham, not phishing versus legitimate email. These results must be described as spam-detection results and only as a demonstration of the phishing-section workflow.

## 1. Install dependencies

Run the notebook from top to bottom. In Colab, no repository clone or folder upload is required.

In [ ]:
%pip install -q joblib matplotlib numpy pandas scikit-learn torch

In [ ]:
import copy
import hashlib
import json
import os
import platform
import random
import re
import shutil
import tarfile
import urllib.request
from collections import Counter
from collections.abc import Iterable
from dataclasses import dataclass
from email import policy
from email.message import Message
from email.parser import BytesParser
from html.parser import HTMLParser
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
from IPython.display import Image, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    classification_report, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, Dataset

def find_local_project_root() -> Path:
    current = Path.cwd().resolve()
    return next(
        (candidate for candidate in [current, *current.parents] if (candidate / "pyproject.toml").is_file()),
        current,
    )

RUNNING_ON_COLAB = Path("/content").is_dir()
WORK_DIR = Path("/content/section_01_workspace") if RUNNING_ON_COLAB else find_local_project_root()
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

config = {
    "seed": 42,
    "data_dir": "data/raw/spamassassin",
    "processed_dir": "data/processed/section_01",
    "models_dir": "models/section_01",
    "results_dir": "reports/section_01",
    "test_fraction": 0.15,
    "validation_fraction": 0.15,
    "classic": {
        "max_features": 20000, "min_document_frequency": 2,
        "max_document_frequency": 0.98, "ngram_min": 1,
        "ngram_max": 2, "max_iterations": 1000,
    },
    "lstm": {
        "max_vocabulary": 20000, "max_sequence_length": 300,
        "embedding_dimension": 128, "hidden_dimension": 128,
        "dropout": 0.3, "batch_size": 64, "epochs": 6,
        "learning_rate": 0.001, "early_stopping_patience": 2,
    },
}

random.seed(config["seed"])
np.random.seed(config["seed"])
print("Environment:", "Google Colab" if RUNNING_ON_COLAB else "Local")
print("Working directory:", WORK_DIR)
print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Download the SpamAssassin corpus

The downloader is implemented directly below. Existing archives and extracted folders are reused.

In [ ]:
SPAMASSASSIN_BASE_URL = "https://spamassassin.apache.org/old/publiccorpus"
SPAMASSASSIN_ARCHIVES = {
    "20030228_easy_ham.tar.bz2": "easy_ham",
    "20030228_spam.tar.bz2": "spam",
}

def safe_extract_tar(archive: tarfile.TarFile, destination: Path) -> None:
    root = destination.resolve()
    for member in archive.getmembers():
        target = (root / member.name).resolve()
        if not target.is_relative_to(root):
            raise ValueError(f"Unsafe archive path: {member.name}")
    archive.extractall(destination)

def download_spamassassin(destination: Path) -> None:
    destination = destination.resolve()
    archive_dir = destination / "archives"
    archive_dir.mkdir(parents=True, exist_ok=True)
    for archive_name, extracted_dir in SPAMASSASSIN_ARCHIVES.items():
        archive_path = archive_dir / archive_name
        if not archive_path.is_file():
            url = f"{SPAMASSASSIN_BASE_URL}/{archive_name}"
            print("Downloading:", url)
            urllib.request.urlretrieve(url, archive_path)
        else:
            print("Using existing archive:", archive_path)
        expected_directory = destination / extracted_dir
        if not expected_directory.is_dir():
            print("Extracting:", archive_path)
            with tarfile.open(archive_path, mode="r:bz2") as archive:
                safe_extract_tar(archive, destination)
        else:
            print("Using existing extracted directory:", expected_directory)
    print("Dataset ready at:", destination)

download_spamassassin(Path(config["data_dir"]))

## 3. Parse, clean, deduplicate, and split emails

The loader extracts the subject and readable MIME body, ignores attachments, converts HTML to visible text, and removes exact duplicates before splitting.

In [ ]:
WHITESPACE_PATTERN = re.compile(r"\s+")

class HTMLTextExtractor(HTMLParser):
    def __init__(self) -> None:
        super().__init__()
        self.fragments: list[str] = []

    def handle_data(self, data: str) -> None:
        self.fragments.append(data)

    def text(self) -> str:
        return " ".join(self.fragments)

@dataclass(frozen=True)
class CorpusSplits:
    train: pd.DataFrame
    validation: pd.DataFrame
    test: pd.DataFrame

def html_to_text(value: str) -> str:
    parser = HTMLTextExtractor()
    parser.feed(value)
    return parser.text()

def mime_part_text(part: Message) -> str:
    try:
        value = part.get_content()
    except (LookupError, UnicodeDecodeError):
        payload = part.get_payload(decode=True) or b""
        value = payload.decode("utf-8", errors="replace")
    return value if isinstance(value, str) else str(value)

def extract_email_text(path: Path) -> str:
    message = BytesParser(policy=policy.default).parsebytes(path.read_bytes())
    subject = str(message.get("subject", ""))
    plain_parts: list[str] = []
    html_parts: list[str] = []
    parts = message.walk() if message.is_multipart() else [message]
    for part in parts:
        if part.is_multipart() or part.get_content_disposition() == "attachment":
            continue
        if part.get_content_type() == "text/plain":
            plain_parts.append(mime_part_text(part))
        elif part.get_content_type() == "text/html":
            html_parts.append(html_to_text(mime_part_text(part)))
    body = " ".join(plain_parts or html_parts)
    return WHITESPACE_PATTERN.sub(" ", f"Subject: {subject} Body: {body}").strip()

def records_from_directory(directory: Path, label: int) -> list[dict[str, object]]:
    records: list[dict[str, object]] = []
    for path in sorted(directory.rglob("*")):
        if not path.is_file() or path.name.lower() == "cmds":
            continue
        text = extract_email_text(path)
        if text:
            records.append({
                "path": str(path), "text": text, "label": label,
                "sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),
            })
    return records

def load_spamassassin_corpus(data_dir: Path) -> pd.DataFrame:
    ham_dir, spam_dir = data_dir / "easy_ham", data_dir / "spam"
    if not ham_dir.is_dir() or not spam_dir.is_dir():
        raise FileNotFoundError("The SpamAssassin archive was not extracted correctly.")
    records = records_from_directory(ham_dir, 0) + records_from_directory(spam_dir, 1)
    frame = pd.DataFrame.from_records(records)
    if frame.empty:
        raise ValueError("No email records were found.")
    return frame.drop_duplicates("sha256").sort_values(["label", "path"]).reset_index(drop=True)

def stratified_splits(frame: pd.DataFrame, test_fraction: float, validation_fraction: float, seed: int) -> CorpusSplits:
    if test_fraction <= 0 or validation_fraction <= 0 or test_fraction + validation_fraction >= 1:
        raise ValueError("Split fractions must be positive and sum to less than one.")
    train_validation, test = train_test_split(
        frame, test_size=test_fraction, random_state=seed, stratify=frame["label"],
    )
    relative_validation = validation_fraction / (1 - test_fraction)
    train, validation = train_test_split(
        train_validation, test_size=relative_validation, random_state=seed,
        stratify=train_validation["label"],
    )
    return CorpusSplits(
        train.reset_index(drop=True), validation.reset_index(drop=True), test.reset_index(drop=True),
    )

def save_split_manifest(splits: CorpusSplits, output_dir: Path) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    parts = []
    for name, frame in (("train", splits.train), ("validation", splits.validation), ("test", splits.test)):
        part = frame[["path", "label", "sha256"]].copy()
        part.insert(0, "split", name)
        parts.append(part)
    target = output_dir / "split_manifest.csv"
    pd.concat(parts, ignore_index=True).to_csv(target, index=False)
    return target

In [ ]:
emails = load_spamassassin_corpus(Path(config["data_dir"]))
label_names = {0: "ham", 1: "spam"}
class_counts = emails["label"].map(label_names).value_counts().reindex(["ham", "spam"])
print("Messages after exact deduplication:", len(emails))
display(class_counts.rename("messages").to_frame())
axis = class_counts.plot.bar(
    color=["#4C78A8", "#E45756"], title="SpamAssassin class distribution",
    ylabel="Messages", rot=0,
)
axis.grid(axis="y", alpha=0.25)
plt.show()

analysis_frame = emails.assign(
    class_name=emails["label"].map(label_names),
    character_count=emails["text"].str.len(),
    word_count=emails["text"].str.split().str.len(),
)
display(analysis_frame.groupby("class_name")[["character_count", "word_count"]].agg(["mean", "median", "min", "max"]).round(1))

In [ ]:
splits = stratified_splits(
    emails, config["test_fraction"], config["validation_fraction"], config["seed"],
)
split_manifest = save_split_manifest(splits, Path(config["processed_dir"]))
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "records": [len(splits.train), len(splits.validation), len(splits.test)],
    "spam_rate": [splits.train["label"].mean(), splits.validation["label"].mean(), splits.test["label"].mean()],
})
display(split_summary.style.format({"spam_rate": "{:.2%}"}))
print("Split manifest:", split_manifest)

## 4. Shared preprocessing and evaluation functions

In [ ]:
URL_PATTERN = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)
EMAIL_PATTERN = re.compile(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b")
NON_WORD_PATTERN = re.compile(r"[^a-z0-9_]+")

def normalize_text(text: str) -> str:
    text = URL_PATTERN.sub(" urltoken ", text.lower())
    text = EMAIL_PATTERN.sub(" emailtoken ", text)
    return WHITESPACE_PATTERN.sub(" ", NON_WORD_PATTERN.sub(" ", text)).strip()

def tokenize(text: str) -> list[str]:
    normalized = normalize_text(text)
    return normalized.split() if normalized else []

def binary_metrics(labels: np.ndarray, probabilities: np.ndarray) -> dict[str, object]:
    predictions = (probabilities >= 0.5).astype(int)
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(labels, probabilities)),
        "average_precision": float(average_precision_score(labels, probabilities)),
        "confusion_matrix": confusion_matrix(labels, predictions).tolist(),
        "classification_report": classification_report(
            labels, predictions, target_names=["ham", "spam"], output_dict=True, zero_division=0,
        ),
    }

def save_metrics(metrics: dict[str, object], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

def save_evaluation_plots(labels: np.ndarray, probabilities: np.ndarray, model_name: str, output_dir: Path) -> list[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    predictions = (probabilities >= 0.5).astype(int)
    safe_name = model_name.lower().replace(" ", "-")
    created = []

    figure, axis = plt.subplots(figsize=(5.5, 4.5))
    ConfusionMatrixDisplay.from_predictions(
        labels, predictions, display_labels=["ham", "spam"], cmap="Blues", colorbar=False, ax=axis,
    )
    axis.set_title(f"{model_name}: confusion matrix")
    figure.tight_layout()
    path = output_dir / f"{safe_name}-confusion-matrix.png"
    figure.savefig(path, dpi=180)
    plt.close(figure)
    created.append(path)

    false_positive_rate, true_positive_rate, _ = roc_curve(labels, probabilities)
    figure, axis = plt.subplots(figsize=(5.5, 4.5))
    axis.plot(false_positive_rate, true_positive_rate, label=f"AUC = {roc_auc_score(labels, probabilities):.3f}")
    axis.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random")
    axis.set(xlabel="False positive rate", ylabel="True positive rate", title=f"{model_name}: ROC curve")
    axis.legend(loc="lower right")
    axis.grid(alpha=0.25)
    figure.tight_layout()
    path = output_dir / f"{safe_name}-roc-curve.png"
    figure.savefig(path, dpi=180)
    plt.close(figure)
    created.append(path)

    precision, recall, _ = precision_recall_curve(labels, probabilities)
    figure, axis = plt.subplots(figsize=(5.5, 4.5))
    axis.plot(recall, precision, label=f"AP = {average_precision_score(labels, probabilities):.3f}")
    axis.set(xlabel="Recall", ylabel="Precision", title=f"{model_name}: precision-recall curve")
    axis.legend(loc="lower left")
    axis.grid(alpha=0.25)
    figure.tight_layout()
    path = output_dir / f"{safe_name}-precision-recall-curve.png"
    figure.savefig(path, dpi=180)
    plt.close(figure)
    created.append(path)
    return created

example_text = "URGENT: Contact Support@Example.com at https://example.com/login!"
print("Original:  ", example_text)
print("Normalized:", normalize_text(example_text))

## 5. Classic model: TF-IDF and Logistic Regression

In [ ]:
def train_classic_model(train: pd.DataFrame, model_config: dict[str, object]) -> Pipeline:
    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            preprocessor=normalize_text, stop_words="english",
            max_features=int(model_config["max_features"]),
            min_df=int(model_config["min_document_frequency"]),
            max_df=float(model_config["max_document_frequency"]),
            ngram_range=(int(model_config["ngram_min"]), int(model_config["ngram_max"])),
            sublinear_tf=True,
        )),
        ("classifier", LogisticRegression(
            max_iter=int(model_config["max_iterations"]), class_weight="balanced", random_state=42,
        )),
    ])
    model.fit(train["text"], train["label"])
    return model

def evaluate_classic_model(model: Pipeline, test: pd.DataFrame, model_dir: Path, results_dir: Path) -> dict[str, object]:
    probabilities = model.predict_proba(test["text"])[:, 1]
    labels = test["label"].to_numpy(dtype=int)
    metrics = binary_metrics(labels, probabilities)
    model_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, model_dir / "tfidf-logistic-regression.joblib")
    save_metrics(metrics, results_dir / "metrics/classic.json")
    save_evaluation_plots(labels, probabilities, "TF-IDF Logistic Regression", results_dir / "figures")
    return metrics

models_dir = Path(config["models_dir"])
results_dir = Path(config["results_dir"])
classic_model = train_classic_model(splits.train, config["classic"])
classic_metrics = evaluate_classic_model(classic_model, splits.test, models_dir, results_dir)
display(pd.Series({
    name: classic_metrics[name] for name in ("accuracy", "precision", "recall", "f1", "roc_auc", "average_precision")
}, name="TF-IDF Logistic Regression").to_frame("score").style.format("{:.4f}"))

In [ ]:
for figure_name in (
    "tf-idf-logistic-regression-confusion-matrix.png",
    "tf-idf-logistic-regression-roc-curve.png",
    "tf-idf-logistic-regression-precision-recall-curve.png",
):
    display(Image(filename=str(results_dir / "figures" / figure_name)))

## 6. Deep-learning model: LSTM

The vocabulary is learned only from training messages. Weighted loss addresses class imbalance, and validation F1 controls checkpoint selection and early stopping.

In [ ]:
PAD_TOKEN, UNKNOWN_TOKEN = "<PAD>", "<UNK>"

@dataclass
class Vocabulary:
    tokens: list[str]

    def __post_init__(self) -> None:
        self.token_to_index = {token: index for index, token in enumerate(self.tokens)}

    @classmethod
    def build(cls, texts: Iterable[str], max_size: int) -> "Vocabulary":
        counts: Counter[str] = Counter()
        for text in texts:
            counts.update(tokenize(text))
        return cls([PAD_TOKEN, UNKNOWN_TOKEN, *[token for token, _ in counts.most_common(max_size - 2)]])

    def encode(self, text: str, max_length: int) -> tuple[list[int], int]:
        unknown = self.token_to_index[UNKNOWN_TOKEN]
        encoded = [self.token_to_index.get(token, unknown) for token in tokenize(text)[:max_length]] or [unknown]
        length = len(encoded)
        encoded.extend([self.token_to_index[PAD_TOKEN]] * (max_length - length))
        return encoded, length

class EmailDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, vocabulary: Vocabulary, max_length: int) -> None:
        self.labels = frame["label"].astype(float).tolist()
        self.examples = [vocabulary.encode(text, max_length) for text in frame["text"]]

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int):
        encoded, length = self.examples[index]
        return (torch.tensor(encoded, dtype=torch.long), torch.tensor(length), torch.tensor(self.labels[index], dtype=torch.float32))

class EmailLSTM(nn.Module):
    def __init__(self, vocabulary_size: int, embedding_dimension: int, hidden_dimension: int, dropout: float) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_dimension, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dimension, hidden_dimension, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_dimension, 1)

    def forward(self, token_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(token_ids)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        return self.output(self.dropout(hidden[-1])).squeeze(1)

def predict_lstm(model: EmailLSTM, loader: DataLoader, device: torch.device) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    probabilities, labels = [], []
    with torch.no_grad():
        for token_ids, lengths, batch_labels in loader:
            logits = model(token_ids.to(device), lengths.to(device))
            probabilities.extend(torch.sigmoid(logits).cpu().numpy().tolist())
            labels.extend(batch_labels.numpy().astype(int).tolist())
    return np.asarray(labels), np.asarray(probabilities)

def save_training_plot(history: list[dict[str, float]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    epochs = [int(item["epoch"]) for item in history]
    figure, primary = plt.subplots(figsize=(6.5, 4.5))
    secondary = primary.twinx()
    primary.plot(epochs, [item["training_loss"] for item in history], marker="o", label="Training loss")
    secondary.plot(epochs, [item["validation_f1"] for item in history], marker="s", color="tab:orange", label="Validation F1")
    primary.set(xlabel="Epoch", ylabel="Training loss")
    secondary.set(ylabel="Validation F1", ylim=(0, 1.02))
    primary.grid(alpha=0.25)
    lines = primary.lines + secondary.lines
    primary.legend(lines, [line.get_label() for line in lines], loc="center right")
    figure.suptitle("LSTM training history")
    figure.tight_layout()
    figure.savefig(path, dpi=180)
    plt.close(figure)

def train_and_evaluate_lstm(train: pd.DataFrame, validation: pd.DataFrame, test: pd.DataFrame, model_config: dict[str, object], seed: int, model_dir: Path, results_dir: Path) -> dict[str, object]:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    vocabulary = Vocabulary.build(train["text"], int(model_config["max_vocabulary"]))
    max_length = int(model_config["max_sequence_length"])
    train_data = EmailDataset(train, vocabulary, max_length)
    validation_data = EmailDataset(validation, vocabulary, max_length)
    test_data = EmailDataset(test, vocabulary, max_length)
    batch_size = int(model_config["batch_size"])
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, generator=generator)
    validation_loader = DataLoader(validation_data, batch_size=batch_size)
    test_loader = DataLoader(test_data, batch_size=batch_size)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = EmailLSTM(
        len(vocabulary.tokens), int(model_config["embedding_dimension"]),
        int(model_config["hidden_dimension"]), float(model_config["dropout"]),
    ).to(device)
    positive_count = float(train["label"].sum())
    positive_weight = torch.tensor([(len(train) - positive_count) / positive_count], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(model_config["learning_rate"]))
    history, best_state, best_f1, stale_epochs = [], None, -1.0, 0
    patience = int(model_config["early_stopping_patience"])
    for epoch in range(1, int(model_config["epochs"]) + 1):
        model.train()
        total_loss, observations = 0.0, 0
        for token_ids, lengths, labels in train_loader:
            token_ids, lengths, labels = token_ids.to(device), lengths.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(token_ids, lengths), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += float(loss.item()) * len(labels)
            observations += len(labels)
        validation_labels, validation_probabilities = predict_lstm(model, validation_loader, device)
        validation_f1 = float(f1_score(validation_labels, validation_probabilities >= 0.5, zero_division=0))
        history.append({"epoch": float(epoch), "training_loss": total_loss / observations, "validation_f1": validation_f1})
        print(f"Epoch {epoch}: loss={history[-1]['training_loss']:.4f}, validation_f1={validation_f1:.4f}")
        if validation_f1 > best_f1:
            best_f1, best_state, stale_epochs = validation_f1, copy.deepcopy(model.state_dict()), 0
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                break
    if best_state is None:
        raise RuntimeError("Training did not produce a checkpoint.")
    model.load_state_dict(best_state)
    labels, probabilities = predict_lstm(model, test_loader, device)
    metrics = binary_metrics(labels, probabilities)
    metrics.update({"best_validation_f1": best_f1, "epochs_completed": len(history), "device": str(device), "vocabulary_size": len(vocabulary.tokens)})
    model_dir.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model_state": {key: value.cpu() for key, value in model.state_dict().items()},
        "vocabulary": vocabulary.tokens,
        "model_config": {"embedding_dimension": int(model_config["embedding_dimension"]), "hidden_dimension": int(model_config["hidden_dimension"]), "dropout": float(model_config["dropout"]), "max_sequence_length": max_length},
    }, model_dir / "lstm.pt")
    save_metrics(metrics, results_dir / "metrics/lstm.json")
    save_evaluation_plots(labels, probabilities, "LSTM", results_dir / "figures")
    save_training_plot(history, results_dir / "figures/lstm-training-history.png")
    return metrics

In [ ]:
# Use None for the configured full run. Set to 1 for a quick pipeline check.
LSTM_EPOCHS_OVERRIDE = None
lstm_config = config["lstm"].copy()
if LSTM_EPOCHS_OVERRIDE is not None:
    lstm_config["epochs"] = LSTM_EPOCHS_OVERRIDE
print("LSTM epochs requested:", lstm_config["epochs"])
lstm_metrics = train_and_evaluate_lstm(
    splits.train, splits.validation, splits.test, lstm_config, config["seed"], models_dir, results_dir,
)
display(pd.Series({
    name: lstm_metrics[name] for name in ("accuracy", "precision", "recall", "f1", "roc_auc", "average_precision", "best_validation_f1", "epochs_completed")
}, name="LSTM").to_frame("value"))

In [ ]:
for figure_name in (
    "lstm-training-history.png", "lstm-confusion-matrix.png",
    "lstm-roc-curve.png", "lstm-precision-recall-curve.png",
):
    display(Image(filename=str(results_dir / "figures" / figure_name)))

## 7. Direct comparison and saved outputs

Both models use the same test records. Accuracy is reported, but precision, recall, F1, ROC AUC, and average precision are essential because spam is the minority class.

In [ ]:
metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
comparison = pd.DataFrame([
    {"model": "TF-IDF Logistic Regression", **{name: classic_metrics[name] for name in metric_names}},
    {"model": "LSTM", **{name: lstm_metrics[name] for name in metric_names}},
])
results_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(results_dir / "model-comparison.csv", index=False)
summary = {
    "records_after_deduplication": len(emails),
    "train_records": len(splits.train),
    "validation_records": len(splits.validation),
    "test_records": len(splits.test),
    "class_counts": {"ham": int((emails["label"] == 0).sum()), "spam": int((emails["label"] == 1).sum())},
    "models_run": comparison["model"].tolist(),
    "lstm_epochs_requested": int(lstm_config["epochs"]),
}
(results_dir / "run-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
display(comparison.style.format({name: "{:.4f}" for name in metric_names}).highlight_max(subset=metric_names, color="#d9ead3"))

## 8. Interpretation and limitations

Discuss false positives, false negatives, class imbalance, model complexity, explainability, and whether the LSTM improvement justifies its cost.

Limitations:

- SpamAssassin is spam-labelled rather than phishing-labelled.
- The corpus is old and may contain collection-specific artefacts.
- Easy ham may make the task artificially simple.
- Exact deduplication does not remove near-duplicate campaigns.
- Production thresholds must reflect the different costs of blocked legitimate messages and missed malicious messages.

## 9. Preserve Colab outputs

Colab storage is temporary. This cell creates a ZIP containing the reports and models for download. Local runs already save directly into the repository.

In [ ]:
if RUNNING_ON_COLAB:
    export_dir = WORK_DIR / "section_01_export"
    if export_dir.exists():
        shutil.rmtree(export_dir)
    shutil.copytree(results_dir, export_dir / "reports")
    shutil.copytree(models_dir, export_dir / "models")
    archive = shutil.make_archive("/content/section_01_results", "zip", root_dir=export_dir)
    print("Download from the Colab Files panel:", archive)
else:
    print("Results saved locally at:", results_dir.resolve())

print("Generated files:")
for path in sorted([*results_dir.rglob("*"), *models_dir.rglob("*")]):
    if path.is_file():
        print(" -", path)